# Plan Output Analysis

Analyze **`terraform plan -json`** (or OpenTofu equivalent) machine-readable UI output.

Expects newline-delimited JSON with `@module: "terraform.ui"` and structured `type` / `hook` / `change` records (for example `refresh_start`, `planned_change`, `resource_drift`).

Capture example:

```bash
terraform plan -json > plan-ui.json
export TERRAFORM_LOG_PATH=plan-ui.json
```

For **`TF_LOG=json`** trace logs, use `plan-log-analysis.ipynb` instead. Run `whatisit.ipynb` if unsure.

In [ ]:
import pandas as pd
import commonlib.prep_plan_output_data as prep_plan_output_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as cfg

In [ ]:
c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)
parsed_records = prep_plan_output_data.read_json_from_file(c.TERRAFORM_LOG_PATH)
normalized_records = prep_plan_output_data.normalize_records(parsed_records)
df = pd.json_normalize(normalized_records)

if df.empty:
    raise ValueError(
        "No terraform.ui plan records found. Capture with `terraform plan -json` "
        "or use plan-log-analysis.ipynb for TF_LOG=json trace logs."
    )

print(sorted(df["type"].drop_duplicates().tolist()))

df_refresh_start = df[df["type"] == "refresh_start"].rename(
    columns={"timestamp": "refresh_start_timestamp"}
)
df_refresh_complete = df[df["type"] == "refresh_complete"].rename(
    columns={"timestamp": "refresh_complete_timestamp"}
)

df_merged_refresh = pd.merge(
    df_refresh_start[
        ["resource_id", "refresh_start_timestamp", "resource", "resource_type", "resource_name"]
    ],
    df_refresh_complete[["resource_id", "refresh_complete_timestamp"]],
    on="resource_id",
)

df_merged_refresh["refresh_start_datetime"] = pd.to_datetime(df_merged_refresh["refresh_start_timestamp"])
df_merged_refresh["refresh_complete_datetime"] = pd.to_datetime(df_merged_refresh["refresh_complete_timestamp"])
df_merged_refresh["time_diff_minutes"] = (
    df_merged_refresh["refresh_complete_datetime"] - df_merged_refresh["refresh_start_datetime"]
).dt.total_seconds() / 60

df_resource_drift = df[df["type"] == "resource_drift"]
df_planned_change = df[df["type"] == "planned_change"]

## Type Analysis

In [ ]:
gencharts.generate_plt_by_resource_type(df, "refresh_start")
gencharts.generate_plt_by_resource_type(df, "refresh_complete")
gencharts.generate_plt_by_resource_type(df, "resource_drift")
gencharts.generate_plt_by_resource_type(df, "planned_change")

## Longest Refresh Times

In [ ]:
df_merged_refresh[
    ["resource", "refresh_start_timestamp", "refresh_complete_timestamp", "time_diff_minutes"]
].copy().sort_values(by="time_diff_minutes", ascending=False).head(20)